<a href="https://colab.research.google.com/github/LinearAlgebruh/PredictingMarchMadness/blob/main/ProjectDataPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import torch
!pip install pyreadr
!pip install rapidfuzz
import pyreadr
from rapidfuzz import process, fuzz



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.8 MB/s eta 0:00:00


In [2]:
def readRSD(url, scratch):
  local = pyreadr.download_file(url, scratch)
  result = pyreadr.read_r(local)
  df = result[None]
  return df

RegSzn2014 = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RegSeasonStats/MMStats_2014.rds", "./2014SeasonStats.rda")
AdvSt2014 = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RatingStats/RatingStats_2014.rds", "./2014RatingStats.rda" )

# ids = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/Teams.rds", "./teamIds.rda")
# df_ids = ids[['TeamID', 'TeamName']]
# df_ids.columns = ['id', 'School']
# df_ids.head()

df_ids = pd.read_csv('https://github.com/LinearAlgebruh/PredictingMarchMadness/raw/refs/heads/main/NCAATeamIds.csv')



In [3]:
# Join without exact matches. Still struggles with abbreviations
def fuzzy_merge(df_left, df_right, key_left, key_right, threshold=60, limit=1):
    """
    Merge data frames on key without exact matches. Used AI for this
    """
    right_names = df_right[key_right].tolist()

    def get_best_match(name):
        match = process.extractOne(
            name, right_names,
            scorer=fuzz.token_sort_ratio,   # handles word-order differences
            score_cutoff=threshold
        )
        return match[0] if match else None

    df_left = df_left.copy()
    df_left['_match_key'] = df_left[key_left].apply(get_best_match)

    merged = df_left.merge(
        df_right,
        left_on='_match_key',
        right_on=key_right,
        how='left'
    ).drop(columns=['_match_key'])

    return merged

dfComplete = fuzzy_merge(RegSzn2014, df_ids, 'School','School')
# dfComplete = fuzzy_merge(dfComplete)
dfComplete.iloc[[167]]

df_with_nans = dfComplete[dfComplete['TeamID'].isna()]
df_with_nans


,Rk,School_x,G,W,L,W.L.,SRS,SOS,W.1,L.1,...,AST,STL,BLK,TOV,PF,TeamID,TeamName,School_y,FirstD1Season,LastD1Season
158,159,Loyola (IL),32,10,22,0.313,-4.75,-1.14,4,14,...,419,176,82,426,622,NaN,NaN,NaN,NaN,NaN
160,161,Loyola (MD),30,11,19,0.367,-10.58,-3.23,6,12,...,336,236,128,357,584,NaN,NaN,NaN,NaN,NaN
166,167,Maryland-Baltimore County,30,9,21,0.300,-14.21,-6.55,5,11,...,355,240,84,450,702,NaN,NaN,NaN,NaN,NaN
175,176,Miami (OH),31,13,18,0.419,-4.37,0.13,8,10,...,387,267,73,420,537,NaN,NaN,NaN,NaN,NaN
205,206,NJIT,29,13,16,0.448,-12.52,-7.48,0,0,...,394,187,104,379,585,NaN,NaN,NaN,NaN,NaN
238,239,Pennsylvania,28,8,20,0.286,-6.59,-0.01,5,9,...,408,163,97,450,633,NaN,NaN,NaN,NaN,NaN
328,329,VMI,35,22,13,0.629,-5.53,-6.33,11,5,...,491,254,218,371,661,NaN,NaN,NaN,NaN,NaN


object


object


NameError: name 'df' is not defined

In [26]:
df = readRSD('https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/Prelim2019_RegularSeasonDetailedResults.rds', './DetailedStats.rda')

# Scrape all box scores for a given team from a single season. I think this is the
# best way to get season level data for us.
def get_team_stats(df, team_id, year=None):
    """
    Returns all stats for a given team from a Detailed Results DataFrame,
    normalized to team/opponent perspective (instead of winner/loser).

    Parameters:
        df      : Detailed Results DataFrame
        team_id : The team's ID to filter by
        year    : (optional) Season year to filter by. If None, returns all years.

    Returns:
        DataFrame with one row per game, stats from the team's perspective
    """

    stat_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

    # --- Games where team won ---
    won = df[(df['WTeamID'] == team_id) & (df['Season'] == year)].copy()
    won['TeamID']      = won['WTeamID']
    won['OpponentID']  = won['LTeamID']
    won['TeamScore']   = won['WScore']
    won['OppScore']    = won['LScore']
    won['Result']      = 1
    # won['Location']    = won['WLoc']
    for col in stat_cols:
        won[col]           = won[f'W{col}']
        won[f'Opp{col}']   = won[f'L{col}']

    # --- Games where team lost ---
    lost = df[(df['LTeamID'] == team_id) & (df['Season'] == year)].copy()
    lost['TeamID']     = lost['LTeamID']
    lost['OpponentID'] = lost['WTeamID']
    lost['TeamScore']  = lost['LScore']
    lost['OppScore']   = lost['WScore']
    lost['Result']     = 0
    # Flip location: if winner was Home, this team was Away, and vice versa
    # lost['Location']   = lost['WLoc'].map({'H': 'A', 'A': 'H', 'N': 'N'})
    for col in stat_cols:
        lost[col]          = lost[f'L{col}']
        lost[f'Opp{col}']  = lost[f'W{col}']

    # --- Combine and select final columns ---
    keep_cols = ['Season', 'DayNum', 'TeamID', 'OpponentID', 'TeamScore',
                 'OppScore', 'Result', 'NumOT'] + \
                stat_cols + [f'Opp{col}' for col in stat_cols]

    team_games = pd.concat([won, lost])[keep_cols]

    team_games = team_games.astype(float)

    team_games = team_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)

    team_games = pd.concat([team_games, team_games.mean(axis=0).to_frame().T], ignore_index=True)


    return team_games

def get_tensor(df):
  df = df.drop(columns=['Season', 'DayNum', 'TeamID', 'OpponentID'])

  stat_tensor = torch.tensor(df.iloc[-1], dtype=torch.float32)
  return stat_tensor


df1104 = get_team_stats(df, 1104, 2014)
df1104.tail()
stats = get_tensor(df1104)
print(stats)
print(df1104.columns.tolist())
df1104.tail()
# df1104.dtypes


tensor([66.7097, 67.2581,  0.3871,  0.0968, 22.7097, 51.8064,  5.7419, 17.2581,
        15.5484, 22.7742,  9.9032, 22.5806, 10.7742, 11.8387,  6.7419,  4.2258,
        19.1613, 22.6774, 53.8387,  5.2903, 17.2258, 16.6129, 22.2258, 11.8710,
        23.0968, 11.9677, 12.5161,  5.6774,  4.0645, 19.0000])
['Season', 'DayNum', 'TeamID', 'OpponentID', 'TeamScore', 'OppScore', 'Result', 'NumOT', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppOR', 'OppDR', 'OppAst', 'OppTO', 'OppStl', 'OppBlk', 'OppPF']


/tmp/ipykernel_3251/2045145579.py:66: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stat_tensor = torch.tensor(df.iloc[-1], dtype=torch.float32)


,Season,DayNum,TeamID,OpponentID,TeamScore,OppScore,Result,NumOT,FGM,FGA,...,OppFGA3,OppFTM,OppFTA,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF
27,2014.0,117.000000,1104.0,1120.000000,73.000000,57.000000,1.000000,0.000000,24.000000,49.000000,...,27.000000,14.000000,21.000000,24.000000,22.000000,9.000000,12.000000,6.000000,4.000000,20.0
28,2014.0,120.000000,1104.0,1246.000000,48.000000,55.000000,0.000000,0.000000,19.000000,50.000000,...,28.000000,18.000000,24.000000,15.000000,26.000000,9.000000,12.000000,4.000000,7.000000,11.0
29,2014.0,124.000000,1104.0,1116.000000,83.000000,58.000000,1.000000,0.000000,29.000000,53.000000,...,25.000000,19.000000,25.000000,8.000000,17.000000,9.000000,15.000000,5.000000,5.000000,20.0
30,2014.0,129.000000,1104.0,1261.000000,56.000000,68.000000,0.000000,0.000000,18.000000,50.000000,...,22.000000,3.000000,9.000000,10.000000,28.000000,19.000000,11.000000,6.000000,4.000000,19.0
31,2014.0,71.451613,1104.0,1279.967742,66.709677,67.258065,0.387097,0.096774,22.709677,51.806452,...,17.225806,16.612903,22.225806,11.870968,23.096774,11.967742,12.516129,5.677419,4.064516,19.0
